In [1]:
!git clone https://github.com/SophiaYifei/MediTriage-LLM.git

Cloning into 'MediTriage-LLM'...
remote: Enumerating objects: 22, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 22 (delta 6), reused 13 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (22/22), 601.85 KiB | 10.56 MiB/s, done.
Resolving deltas: 100% (6/6), done.


In [24]:
from datasets import load_dataset
import pandas as pd
import numpy as np
import json
from openai import OpenAI

In [3]:
ds = load_dataset("curaihealth/medical_questions_pairs")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/314k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3048 [00:00<?, ? examples/s]

In [5]:
ds = ds['train']
print(ds)

Dataset({
    features: ['dr_id', 'question_1', 'question_2', 'label'],
    num_rows: 3048
})


In [12]:
df = ds.to_pandas()

df_question_1 = df[['question_1']]
df_question_2 = df[['question_2']]

#display(df_question_1.head())
print(len(df_question_1))
#display(df_question_2.head())
print(len(df_question_2))

df_question_1 = df_question_1.rename(columns={'question_1': 'questions'})
df_question_2 = df_question_2.rename(columns={'question_2': 'questions'})

# Concatenate the dataframes vertically
df_questions = pd.concat([df_question_1, df_question_2], ignore_index=True)

display(df_questions.head())
print(len(df_questions))

df_questions.drop_duplicates(inplace=True)
print(len(df_questions))


3048
3048


,questions
0,After how many hour from drinking an antibioti...
1,After how many hour from drinking an antibioti...
2,Am I over weight (192.9) for my age (39)?
3,Am I over weight (192.9) for my age (39)?
4,Aspirin allergy - is it worth getting a bracelet?


6096
4567


In [14]:
%cd MediTriage-LLM/data/raw

/content/MediTriage-LLM/data


In [19]:
with open('patient_messages.json', 'r') as f:
    data = json.load(f)

df_patient_messages = pd.DataFrame([item['patient_message'] for item in data], columns=['patient_message'])

display(df_patient_messages.head())
print(len(df_patient_messages))

,patient_message
0,I've been noticing intermittent palpitations o...
1,so my back been hurting alot lately like I lif...
2,I've been noticing that my lower back has been...
3,my sleep has been real bad lately idk how to f...
4,is it normal to have really bad heartburn ever...


3000


In [22]:
df_patient_messages = df_patient_messages.rename(columns={'patient_message': 'questions'})

df_input = pd.concat([df_questions, df_patient_messages], ignore_index=True)

display(df_input.head())
print(len(df_input))

,questions
0,After how many hour from drinking an antibioti...
1,Am I over weight (192.9) for my age (39)?
2,Aspirin allergy - is it worth getting a bracelet?
3,"At a doctor's visit, I hit my head against a b..."
4,Been on antibiotics 4 5wks top high tooth dent...


7567
Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


In [28]:
from google.colab import userdata

# 1. Setup API Keys and Clients
# Please add your OpenAI API key to the "Secrets" tab (key icon) on the left panel with the name "OPENAI_API_KEY"
try:
    api_key = userdata.get('OPENAI_API_KEY')
    if api_key is None:
        print("Warning: OPENAI_API_KEY not found in secrets.")
except Exception:
    api_key = "PASTE_YOUR_OPENAI_API_KEY_HERE" # Fallback if secrets are not used

client = OpenAI(api_key=api_key)

In [41]:
check_df = df_input.copy()

# 3. Define the Strict System Prompt
system_prompt = """
You are an expert medical triage system. Read the following patient text and extract the information into a strict JSON format.
You must output ONLY valid JSON.
The JSON must have the following keys:
- "department": The most relevant medical department.
- "accepted_department": The top three most relevant medical departments.
- "symptoms": A list of physical or mental symptoms the patient is currently experiencing (e.g., pain, rash, headache). If none are mentioned, output [].
- "condition": Any diagnosed diseases, chronic conditions, or general medical conditions mentioned in the text (e.g., Hypertension, Diabetes, Asthma), regardless of whether the patient states they have it or is just asking about it. If none, output "Unknown".
- "sentiment": The emotional state of the patient.
- "urgency_level": The urgency level displayed in the text: "Low", "Medium", or "High".
"""

# 4. The API Loop (The "Teacher" Distillation)
data = []

print("Starting API generation loop...")
for index, row in check_df.iterrows():
    raw_patient_text = row['questions'] # Adjust column name based on the specific dataset

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            response_format={ "type": "json_object" }, # Forces the model to output valid JSON
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": raw_patient_text}
            ]
        )

        # Extract the JSON string from the response
        teacher_json_output = response.choices[0].message.content

        df_input.at[index, 'output'] = teacher_json_output

        # Format it into the exact JSONL structure needed for fine-tuning
        fine_tuning_example = {
            "messages": [
                {"role": "system", "content": "Extract the patient triage data into JSON."},
                {"role": "user", "content": raw_patient_text},
                {"role": "assistant", "content": teacher_json_output}
            ]
        }

        data.append(fine_tuning_example)
        print(index, ":", raw_patient_text)
        print(teacher_json_output)

    except Exception as e:
        print(f"Failed on row {index}: {e}")

# 5. Save locally to a JSON file
output_filename = "finetuning_data.json"
with open(output_filename, 'w') as outfile:
    for entry in data:
        json.dump(entry, outfile)
        outfile.write('\n')
"""
output_filename = "medical_triage_finetune_data.jsonl"
with open(output_filename, 'w') as outfile:
    for entry in data:
        json.dump(entry, outfile)
        outfile.write('\n')
"""
print(f"Successfully generated {len(data)} examples.")

Streaming output truncated to the last 5000 lines.
  "urgency_level": "Medium"
}
7210 : hey doc my dad who's 81 has been having really bad stomach ache the last few days hes been taking his meds like usual but it’s not getting better he keeps saying hes fine but i know hes not. should i bring him in or u think its ok to wait? thx
{
  "department": "Gastroenterology",
  "accepted_department": [
    "Gastroenterology",
    "Internal Medicine",
    "Geriatrics"
  ],
  "symptoms": [
    "stomach ache"
  ],
  "condition": "Unknown",
  "sentiment": "Concerned",
  "urgency_level": "Medium"
}
7211 : hey doc, so ive been noticing my hands r kinda tingly sometimes mainly at night not sure if its anything serious or just stress. also my fingers feel kinda stiff in the mornings. its been a few weeks now, any advice? thx!
{
  "department": "Neurology",
  "accepted_department": [
    "Neurology",
    "Rheumatology",
    "Orthopedics"
  ],
  "symptoms": [
    "tingling in hands",
    "stiff fingers"


In [42]:
display(df_input.head(20))

,questions,output
0,After how many hour from drinking an antibioti...,"{\n ""department"": ""Pharmacology"",\n ""accepte..."
1,Am I over weight (192.9) for my age (39)?,"{\n ""department"": ""Nutrition"",\n ""accepted_d..."
2,Aspirin allergy - is it worth getting a bracelet?,"{\n ""department"": ""Allergy and Immunology"",\n..."
3,"At a doctor's visit, I hit my head against a b...","{\n ""department"": ""Infectious Disease"",\n ""a..."
4,Been on antibiotics 4 5wks top high tooth dent...,"{\n ""department"": ""Dentistry"",\n ""accepted_d..."
5,Can Adderall (dextroamphetamine and racemic am...,"{\n ""department"": ""Neurology"",\n ""accepted_d..."
6,Can coarctation of the aorta cause poor growth...,"{\n ""department"": ""Cardiology"",\n ""accepted_..."
7,Can doxycycline treat an ear infection?,"{\n ""department"": ""Otolaryngology"",\n ""accep..."
8,Can vinegar help drain the sinus? I've been ea...,"{\n ""department"": ""Otolaryngology"",\n ""accep..."
9,Could chiari malformation cause pituitary prob...,"{\n ""department"": ""Neurology"",\n ""accepted_d..."


In [44]:
import json
import pandas as pd

# Extract and parse the JSON outputs from the data list
parsed_outputs = []
for entry in data:
    # The assistant's JSON output is in the third message
    json_str = entry["messages"][2]["content"]
    try:
        parsed_outputs.append(json.loads(json_str))
    except json.JSONDecodeError:
        print("Failed to parse a JSON string.")

# Create a DataFrame for easy analysis
df_parsed = pd.DataFrame(parsed_outputs)

# Analyze the requested fields
fields_to_analyze = ['department', 'sentiment', 'urgency_level', 'condition']

for field in fields_to_analyze:
    if field in df_parsed.columns:
        # Convert lists to tuples to make them hashable for unique() and value_counts()
        hashable_series = df_parsed[field].apply(lambda x: tuple(x) if isinstance(x, list) else x)

        print(f"================ {field.upper()} ================")
        print(f"Unique values: {hashable_series.unique().tolist()}")
        print("\nDistribution:")
        print(hashable_series.value_counts())
        print("\n")
    else:
        print(f"Field '{field}' not found in the output JSONs.")

================ DEPARTMENT ================
Unique values: ['Pharmacology', 'Nutrition', 'Allergy and Immunology', 'Infectious Disease', 'Dentistry', 'Neurology', 'Cardiology', 'Otolaryngology', 'Dermatology', 'Pediatrics', 'Endocrinology', 'Genetics', 'Obstetrics and Gynecology', 'Psychiatry', 'Orthopedics', 'Urology', 'Mental Health', 'Pharmacy', 'Rheumatology', 'Gastroenterology', 'Gynecology', 'Obstetrics', 'Colorectal Surgery', 'Physical Medicine and Rehabilitation', 'Oral Surgery', 'Oncology', 'Breast Health', 'Neurosurgery', 'Nephrology', 'Toxicology', 'Ophthalmology', 'Hematology', 'Vascular Surgery', 'Internal Medicine', 'Orthodontics', 'Radiology', 'Infectious Diseases', 'Emergency Medicine', 'Plastic Surgery', 'Dental', 'Oral and Maxillofacial Surgery', 'Obstetrics/Gynecology', 'Physical Therapy', 'Pulmonology', 'Proctology', 'Reproductive Endocrinology', 'General Surgery', 'Oral Health', 'Oral Medicine', 'Nutrition and Exercise', 'ENT', 'Respiratory Therapy', 'Vascular Med

In [45]:
# Save df_input to a CSV file
df_input.to_csv('df_input.csv', index=False)
print("df_input successfully saved to 'df_input.csv'")

df_input successfully saved to 'df_input.csv'


In [46]:
from google.colab import files

# Download the CSV file
files.download('df_input.csv')

# Download the JSON file
files.download('finetuning_data.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>